# AdamW LR Doğrulaması — 24 Koşu

Amaç: AdamW'yi **weight_decay=0.01** ile gerçekten AdamW olarak çalıştırıp dört LR adayını altı aktivasyonun tamamında yeniden doğrulamak.

**6 aktivasyon × 4 LR = 24 koşu**, her biri 30 epoch, seed 42. CIFAR-10 validation kullanılır; test setine bakılmaz.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


## Güncel GitHub sürümünü al


In [ ]:
import io, os, shutil, tarfile, requests, json
from pathlib import Path

OWNER='erdemci07'; REPO='derin-ogrenme-egitim-dinamikleri'; BRANCH='main'
repo_dir=Path('/content/derin-ogrenme-egitim-dinamikleri')
extract_root=Path('/content/_tez_repo_extract')
shutil.rmtree(repo_dir, ignore_errors=True)
shutil.rmtree(extract_root, ignore_errors=True)
extract_root.mkdir(parents=True, exist_ok=True)
r=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/{BRANCH}', timeout=60)
r.raise_for_status(); commit_sha=r.json()['sha']
a=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/tarball/{commit_sha}', timeout=120)
a.raise_for_status()
with tarfile.open(fileobj=io.BytesIO(a.content), mode='r:gz') as tar:
    tar.extractall(extract_root)
kokler=[p for p in extract_root.iterdir() if p.is_dir()]
if len(kokler)!=1:
    raise RuntimeError('GitHub arşiv yapısı beklenenden farklı.')
shutil.move(str(kokler[0]), str(repo_dir)); os.chdir(repo_dir)
os.environ['TEZ_GIT_COMMIT']=commit_sha
print('Repo hazır:', commit_sha)


## Drive yolları


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_KOK=Path('/content/drive/MyDrive')
DRIVE_VERI=DRIVE_KOK/'tez_datasets'
DRIVE_PROJE=DRIVE_KOK/'tez_deney_sonuclari'
DRIVE_AYAR=DRIVE_PROJE/'ortak_ayarlar'
DRIVE_ADAMW=DRIVE_PROJE/'adamw_wd001_lr_pilotu'
DRIVE_ADAMW_ANALIZ=DRIVE_PROJE/'adamw_wd001_lr_analiz'
if not DRIVE_VERI.is_dir():
    raise FileNotFoundError(f'Veri klasörü bulunamadı: {DRIVE_VERI}')
DRIVE_AYAR.mkdir(parents=True, exist_ok=True)
DRIVE_ADAMW.mkdir(parents=True, exist_ok=True)
DRIVE_ADAMW_ANALIZ.mkdir(parents=True, exist_ok=True)
SPLIT_DOSYASI=DRIVE_AYAR/'cifar10_split.json'
STATS_DOSYASI=DRIVE_AYAR/'cifar10_stats.json'
os.environ['TEZ_VERI_KLASORU']=str(DRIVE_VERI)
os.environ['TEZ_SPLIT_DOSYASI']=str(SPLIT_DOSYASI)
os.environ['TEZ_STATS_DOSYASI']=str(STATS_DOSYASI)
os.environ['TEZ_CIKTI_KLASORU']=str(DRIVE_ADAMW)
print('AdamW pilot çıktıları:', DRIVE_ADAMW)


## CUDA ve bağımlılıklar


In [ ]:
subprocess.run(['python','-m','pip','install','-q','-r','requirements_colab.txt'], check=True)
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
if not torch.cuda.is_available():
    raise RuntimeError('T4/CUDA aktif değil.')


## 24 koşuyu çalıştır / devam ettir

Tamamlanan koşular atlanır; NaN/Inf üreten koşular geçersiz aday olarak işaretlenir ve yeniden çalıştırılmaz.


In [ ]:
import importlib
import calistir_lr_pilotu as lr
importlib.reload(lr)
lr.ana(['--calistir','--ayarlar-dosyasi','ayarlar/adamw_wd001_lr_pilotu.yaml'])


## 24/24 tamamlandıysa LR sonucunu çıkar


In [ ]:
import math, statistics

tamam={}; kararsiz={}
for p in DRIVE_ADAMW.glob('*/tamamlandi.json'):
    try:
        d=json.loads(p.read_text(encoding='utf-8')); tamam[d['run_id']]=d
    except Exception:
        pass
for p in DRIVE_ADAMW.glob('*/kararsiz.json'):
    try:
        d=json.loads(p.read_text(encoding='utf-8'))
        if d['run_id'] not in tamam: kararsiz[d['run_id']]=d
    except Exception:
        pass

deger=len(tamam)+len(kararsiz)
print(f'Değerlendirilen: {deger}/24 ({len(tamam)} sağlıklı, {len(kararsiz)} kararsız/uygunsuz)')

if deger == 24:
    lrler=[0.0001,0.0003,0.001,0.003]
    satirlar=[]
    for aday in lrler:
        rs=[d for d in tamam.values() if d['optimizer']=='adamw' and math.isclose(float(d['learning_rate']),aday,rel_tol=0,abs_tol=1e-15)]
        uygun=len(rs)==6
        if uygun:
            acc=[float(d['final_validation_accuracy']) for d in rs]
            loss=[float(d['final_validation_loss']) for d in rs]
            satirlar.append({
                'learning_rate':aday,'uygun':True,'aktivasyon_sayisi':6,
                'mean_final_validation_accuracy':statistics.mean(acc),
                'std_final_validation_accuracy':statistics.pstdev(acc),
                'mean_final_validation_loss':statistics.mean(loss),
                'std_final_validation_loss':statistics.pstdev(loss),
            })
        else:
            satirlar.append({'learning_rate':aday,'uygun':False,'aktivasyon_sayisi':len(rs)})

    uygunlar=[x for x in satirlar if x['uygun']]
    if not uygunlar:
        raise RuntimeError('Altı aktivasyonda da sağlıklı kalan AdamW LR adayı yok.')
    uygunlar.sort(key=lambda x:(-x['mean_final_validation_accuracy'],x['mean_final_validation_loss'],x['learning_rate']))
    secilen=uygunlar[0]
    sonuc={
        'optimizer':'adamw','weight_decay':0.01,'seed':42,'epoch':30,
        'protokol':'önce en yüksek 30. epoch ortalama validation accuracy; sonra en düşük validation loss; tam eşitlikte düşük LR',
        'adaylar':satirlar,'secilen_learning_rate':secilen['learning_rate'],
        'tamamlanan_kosu':len(tamam),'kararsiz_kosu':len(kararsiz)
    }
    (DRIVE_ADAMW_ANALIZ/'adamw_wd001_lr_sonucu.json').write_text(json.dumps(sonuc,ensure_ascii=False,indent=2),encoding='utf-8')
    print('\nAdamW aday özeti:')
    for x in satirlar:
        if x['uygun']:
            print(f"LR={x['learning_rate']} | mean val acc={x['mean_final_validation_accuracy']:.6f} | mean val loss={x['mean_final_validation_loss']:.6f}")
        else:
            print(f"LR={x['learning_rate']} | UYGUN DEĞİL | sağlıklı aktivasyon={x['aktivasyon_sayisi']}/6")
    print('\nSEÇİLEN AdamW LR =', secilen['learning_rate'])
    print('Sonuç dosyası:', DRIVE_ADAMW_ANALIZ/'adamw_wd001_lr_sonucu.json')
else:
    print('Henüz 24/24 değil. Kesinti olursa aynı notebooku yeniden Run all yapın.')
